# Ablation experiments

Goal:
Compare how different feature and preprocessing choices affect model quality.

We will compare several variants:

1. all features with missing indicators
2. all features without missing indicators
3. without Insulin
4. without SkinThickness
5. without Insulin and SkinThickness
6. without Pregnancies

For each variant we will evaluate:

- Logistic Regression
- k-NN with n_neighbors = 15
- Decision Tree with max_depth = 4

All comparisons are performed using cross-validation on X_train only.

The final X_test set is not used for model selection.

In [7]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, precision_score, recall_score, f1_score

from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_utils import load_data, split_features_target, make_train_test_split

from src.preprocessing import (
    replace_suspicious_zeros,
    replace_suspicious_zeros_with_indicators,
)

from src.preprocessing import (
    replace_suspicious_zeros,
    replace_suspicious_zeros_with_indicators,
)

In [8]:
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "diabetes.csv"

df = load_data(DATA_PATH)

X, y = split_features_target(df)

X_train, X_test, y_train, y_test = make_train_test_split(X, y)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("\nTrain target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest target distribution:")
print(y_test.value_counts(normalize=True))

X_train shape: (614, 8)
X_test shape: (154, 8)

Train target distribution:
Outcome
0    0.651466
1    0.348534
Name: proportion, dtype: float64

Test target distribution:
Outcome
0    0.649351
1    0.350649
Name: proportion, dtype: float64


In [9]:
X_train_with_indicators = replace_suspicious_zeros_with_indicators(X_train)

print("Original X_train shape:", X_train.shape)
print("With indicators shape:", X_train_with_indicators.shape)
print(X_train_with_indicators.columns.tolist())
print(X_train_with_indicators[["Insulin_missing", "SkinThickness_missing"]].mean())

Original X_train shape: (614, 8)
With indicators shape: (614, 10)
['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Insulin_missing', 'SkinThickness_missing']
Insulin_missing          0.472313
SkinThickness_missing    0.285016
dtype: float64


In [10]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

scoring = {
    "accuracy": "accuracy",
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": make_scorer(recall_score, zero_division=0),
    "f1": make_scorer(f1_score, zero_division=0),
    "roc_auc": "roc_auc",
}

In [11]:
feature_variants = {
    "all_with_indicators": {
        "drop_cols": [],
        "use_indicators": True,
    },
    "all_without_indicators": {
        "drop_cols": [],
        "use_indicators": False,
    },
    "without_insulin": {
        "drop_cols": ["Insulin"],
        "use_indicators": True,
    },
    "without_skin_thickness": {
        "drop_cols": ["SkinThickness"],
        "use_indicators": True,
    },
    "without_insulin_and_skin_thickness": {
        "drop_cols": ["Insulin", "SkinThickness"],
        "use_indicators": True,
    },
    "without_pregnancies": {
        "drop_cols": ["Pregnancies"],
        "use_indicators": True,
    },
}

In [12]:
model_configs = {
    "logreg": {
        "model": LogisticRegression(max_iter=1000),
        "use_scaler": True,
    },
    "knn_k15": {
        "model": KNeighborsClassifier(n_neighbors=15),
        "use_scaler": True,
    },
    "tree_depth4": {
        "model": DecisionTreeClassifier(max_depth=4, random_state=42),
        "use_scaler": False,
    },
}

In [13]:
def build_pipeline(model, use_scaler: bool, use_indicators: bool) -> Pipeline:
    """Build sklearn pipeline for one preprocessing/model configuration."""
    zero_handler_func = (
        replace_suspicious_zeros_with_indicators
        if use_indicators
        else replace_suspicious_zeros
    )

    steps = [
        ("zero_handler", FunctionTransformer(zero_handler_func, validate=False)),
        ("imputer", SimpleImputer(strategy="median")),
    ]

    if use_scaler:
        steps.append(("scaler", StandardScaler()))

    steps.append(("model", model))

    return Pipeline(steps=steps)

In [14]:
def evaluate_variant_model(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    variant_name: str,
    variant_config: dict,
    model_name: str,
    model_config: dict,
) -> dict:
    """Evaluate one feature variant and one model using cross-validation."""
    drop_cols = variant_config["drop_cols"]
    use_indicators = variant_config["use_indicators"]

    X_variant = X_train.drop(columns=drop_cols)

    pipeline = build_pipeline(
        model=model_config["model"],
        use_scaler=model_config["use_scaler"],
        use_indicators=use_indicators,
    )

    cv_results = cross_validate(
        pipeline,
        X_variant,
        y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=False,
    )

    return {
        "variant": variant_name,
        "model": model_name,
        "accuracy_mean": cv_results["test_accuracy"].mean(),
        "accuracy_std": cv_results["test_accuracy"].std(),
        "precision_mean": cv_results["test_precision"].mean(),
        "recall_mean": cv_results["test_recall"].mean(),
        "f1_mean": cv_results["test_f1"].mean(),
        "roc_auc_mean": cv_results["test_roc_auc"].mean(),
    }

In [15]:
ablation_results = []

for variant_name, variant_config in feature_variants.items():
    for model_name, model_config in model_configs.items():
        result = evaluate_variant_model(
            X_train=X_train,
            y_train=y_train,
            variant_name=variant_name,
            variant_config=variant_config,
            model_name=model_name,
            model_config=model_config,
        )
        ablation_results.append(result)

ablation_results_df = pd.DataFrame(ablation_results)

ablation_results_df.sort_values("f1_mean", ascending=False)

,variant,model,accuracy_mean,accuracy_std,precision_mean,recall_mean,f1_mean,roc_auc_mean
6,without_insulin,logreg,0.793109,0.019565,0.767990,0.588704,0.665366,0.844516
9,without_skin_thickness,logreg,0.791483,0.020198,0.761634,0.593355,0.664675,0.845215
0,all_with_indicators,logreg,0.789871,0.014476,0.760518,0.588815,0.661436,0.843462
12,without_insulin_and_skin_thickness,logreg,0.788231,0.019057,0.758018,0.584164,0.658404,0.847216
3,all_without_indicators,logreg,0.788231,0.019057,0.763808,0.574751,0.654545,0.843364
15,without_pregnancies,logreg,0.781687,0.026615,0.738524,0.583942,0.651359,0.838636
4,all_without_indicators,knn_k15,0.768733,0.032735,0.718961,0.570653,0.631865,0.831831
1,all_with_indicators,knn_k15,0.773691,0.028335,0.751531,0.537763,0.623121,0.821378
16,without_pregnancies,knn_k15,0.763841,0.026729,0.710131,0.551384,0.619615,0.827349
13,without_insulin_and_skin_thickness,knn_k15,0.755658,0.032046,0.710311,0.537763,0.605478,0.830379


In [16]:
ablation_results_df.sort_values("recall_mean", ascending=False).head(5)

,variant,model,accuracy_mean,accuracy_std,precision_mean,recall_mean,f1_mean,roc_auc_mean
9,without_skin_thickness,logreg,0.791483,0.020198,0.761634,0.593355,0.664675,0.845215
0,all_with_indicators,logreg,0.789871,0.014476,0.760518,0.588815,0.661436,0.843462
6,without_insulin,logreg,0.793109,0.019565,0.767990,0.588704,0.665366,0.844516
12,without_insulin_and_skin_thickness,logreg,0.788231,0.019057,0.758018,0.584164,0.658404,0.847216
15,without_pregnancies,logreg,0.781687,0.026615,0.738524,0.583942,0.651359,0.838636


Insulin можно рассматривать как кандидат на удаление в финальной Logistic Regression pipeline.

In [ ]:
1. logreg + all_with_indicators
   базовый сильный вариант

2. logreg + without_insulin
   лучший F1

3. logreg + without_skin_thickness
   лучший recall